# NHS Adult Autism Assessment Pathway DES Model

Interactive demonstration of the **`des`** discrete-event simulation model.

## Prerequisites

Create and activate the Conda environment:

```bash
conda env create -f environment.yml
conda activate sim_env
```

Open this notebook from within the project and run all cells from top to bottom.

The modules can then be imported directly, for example:

```python
from des.model.experiment import Experiment
from des.model.patient import Patient
```

## Table of Contents

---

## Table of Contents

1. [Background](#1-background)
2. [Project Aims](#2-project-aims)
3. [Proposed Solution](#3-proposed-solution)
4. [Discrete Event Simulation (DES) Model](#4-discrete-event-simulation-des-model)
   - 4.1 [Overview](#41-overview-of-the-des-model)
   - 4.2 [Default Parameters](#42-default-parameters)
   - 4.3 [Experiment Class](#43-experiment-class)
   - 4.4 [Patient Class](#44-patient-class)
   - 4.5 [Workforce Resource](#45-workforce-resource-and-resource-constraints)
   - 4.6 [Autism Pathway System Class](#46-autism-pathway-system-class)
5. [Classes and Functions](#5-classes-and-functions-of-the-autism-pathway-model)
6. [Simulation Execution](#6-simulation-execution)
   - 6.1 [Single Trace Run](#61-single-trace-run)
   - 6.2 [Single Simulation Run — drain and warm-up](#62-single-simulation-run)
   - 6.3 [Multiple Replications](#63-multiple-replications-fixed-and-random-seeds)
7. [Model Applications](#7-model-applications)
   - 7.1–7.6 Baseline, experiments, warm-up, calibration, freeze-state, scenarios
8. [Model Verification and Validation](#8-model-verification-and-validation)
9. [Future Work](#9-future-work)


---

# 1. Background

Adult Autism Assessment services across the NHS experience substantial waiting lists and increasing referral demand. Many services operate under limited clinical capacity, resulting in long Referral-to-Treatment (RTT) times, growing assessment backlogs, and increasing pressure on specialist clinicians.

Evaluating operational interventions directly within NHS services is difficult because of financial costs, service disruption, and clinical risk.

Discrete Event Simulation (DES) provides a virtual environment where service behaviour can be reproduced, analysed, and tested before operational changes are implemented.


---

# 2. Project Aims

The objectives of this project are to:

- Develop a realistic DES model of the NHS Adult Autism Assessment pathway.
- Model workforce resource constraints using clinician working hours.
- Represent patient flow from referral to discharge.
- Capture waiting lists and Referral-to-Treatment (RTT) performance.
- Support configurable NHS providers using external configuration files.
- Produce realistic operational system states.
- Support intervention and scenario analysis.



---
# 3. Proposed Solution

The project proposes a configurable **Discrete Event Simulation (DES)** framework developed using **SimPy** to model neurodevelopmental (ADHD/ASD) assessment pathways.

The framework consists of:

### Current implementation

- Patient-level discrete event simulation
- Workforce-hour constrained resource model
- NHS autism pathway representation
- Warm-up, collection, drain simulation phases
- Scenario analysis from configurable model parameters
- Comprehensive Verification and Validation (V&V) framework

### Planned extensions

- Provider-specific calibration to reproduce the current operational state of an NHS provider *(see §7.4)*
- Intervention analysis from the calibrated provider state to evaluate capacity changes, demand changes, and recovery trajectories *(see §7.5)*

The same simulation engine can therefore be configured for different NHS providers without modifying the underlying DES logic.

---

# 4. Discrete Event Simulation (DES) Model

## 4.1 Overview of the DES Model

- Built using **SimPy**.
- Event-driven patient simulation.
- Individual patient entities.
- Multiple clinical pathway stages.
- Workforce-hour constrained clinical resources.
- Priority and standard waiting queues.
- Daily replenishment of clinician hours.
- **Warm-up → collection → drain** phases.
- configurable parameters.

### Patient flow

Each patient is a SimPy process. Referrals arrive on **weekdays only**. Capacity-constrained stages queue for clinician **hours**.



#### Figure 1 — Main pathway
![Figure 1: Patient pathway](figures/01_patient_pathway.png)

#### Figure 2 — Clinical stage 
![Figure 2: Clinical stage](figures/02_clinical_stage.png)

#### Figure 3 — Workforce
![Figure 3: Review loop](figures/03_workforce_hours.png)


In [53]:
import pandas as pd
from des.model import (
    Audit,
    AutismPathwaySystem,
    Experiment,
    Patient,
    WorkforceHoursResource,
    multiple_runs,
    single_run,
)
from des.model.parameters import (
    DEFAULT_RND_SET,
    MAX_DRAIN_DAYS,
    N_REP,
    REFERRALS_PER_DAY,
    RUN_LENGTH,
    WARMUP_DAYS,
)


## 4.2 Default Parameters

Default values are defined in `adhd_simpy/Model/parameters.py`.

| Category | Parameters | Description |
|----------|------------|-------------|
| **Simulation Control** | `RUN_LENGTH`, `WARMUP_DAYS`, `COOLDOWN_DAYS`, `NUMBER_OF_RUNS`, `TRACE`, `DEFAULT_SEED` | Controls simulation duration, warm-up, replications, tracing and random seed. |
| **Referral Generation** | `MEAN_REFERRAL_INTERVAL`, `PRIORITY_REFERRAL_PROPORTION` | Controls referral arrival process and priority referral proportion. |
| **Clinical Pathway Probabilities** | `P_TRIAGE_REJECTION`, `P_SCREENING_DISCHARGE`, `P_PREASSESSMENT_REJECTION`, `P_FURTHER_ASSESSMENT`, `P_AUTISM_DIAGNOSIS`, `P_POST_DIAG_SUPPORT`, `P_REVIEW`, `P_SELF_DISCHARGE` | Defines branching probabilities throughout the autism assessment pathway. |
| **Service Duration Parameters** | `SCREENING_DURATION`, `PREASSESSMENT_DURATION`, `ASSESSMENT_DURATION`, `FURTHER_ASSESSMENT_DURATION`, `POST_DIAG_DURATION`, `OTHER_DURATION`, `REVIEW_DURATION` | Probability distributions defining appointment durations at each clinical stage. |
| **Workforce Capacity** | `SCREENING_WORKFORCE_HOURS`, `PREASSESSMENT_WORKFORCE_HOURS`, `ASSESSMENT_WORKFORCE_HOURS`, `FURTHER_ASSESSMENT_WORKFORCE_HOURS`, `POST_DIAG_WORKFORCE_HOURS`, `OTHER_WORKFORCE_HOURS`, `REVIEW_WORKFORCE_HOURS` | Daily clinician hours available at each pathway stage. |
| **Statistical Configuration** | Collection period, confidence interval settings, random number streams | Controls statistical analysis and reproducibility of simulation experiments. |

These parameters allow the same DES engine to be configured for different NHS providers without modifying the underlying simulation model.

In [54]:
from des.model import parameters as p

default_params = pd.Series(
    {
        "REFERRALS_PER_DAY": p.REFERRALS_PER_DAY,
        "WARMUP_DAYS": p.WARMUP_DAYS,
        "RUN_LENGTH": p.RUN_LENGTH,
        "MAX_DRAIN_DAYS": p.MAX_DRAIN_DAYS,
        "PCT_REFERRAL_REJECTED": p.PCT_REFERRAL_REJECTED,
        "PCT_NON_DIAGNOSIS_AT_ASSESSMENT": p.PCT_NON_DIAGNOSIS_AT_ASSESSMENT,
        "WORKFORCE_HOURS_ASSESSMENT": p.WORKFORCE_HOURS_ASSESSMENT,
        "PCT_PRIORITY_ASSESSMENT": p.PCT_PRIORITY_ASSESSMENT,
        "DEFAULT_RND_SET": p.DEFAULT_RND_SET,
        "N_REP": p.N_REP,
    },
    name="default",
)
default_params


REFERRALS_PER_DAY                     5.00
WARMUP_DAYS                         730.00
RUN_LENGTH                         1825.00
MAX_DRAIN_DAYS                     3650.00
PCT_REFERRAL_REJECTED                 0.20
PCT_NON_DIAGNOSIS_AT_ASSESSMENT       0.20
WORKFORCE_HOURS_ASSESSMENT           10.00
PCT_PRIORITY_ASSESSMENT               0.15
DEFAULT_RND_SET                      42.00
N_REP                                20.00
Name: default, dtype: float64

## 4.3 Experiment Class

**Module:** `des/model/experiment.py`

Responsible for:

- Simulation configuration
- Random number management (25 independent streams)
- Parameter loading (via constructor `**kwargs` overrides)
- Results collection (`results` flow counters)
- Seeded distribution objects for arrivals, durations, and branching

```python
experiment = Experiment(auditor=Audit(), random_number_set=42)
experiment = Experiment(auditor=Audit(), iat=1/10, workforce_hours_assessment=36)
```


## 4.4 Patient Class

**Module:** `des/model/patient.py`

Responsible for modelling:

- Individual referrals as SimPy processes
- Clinical pathway progression (triage → 7 stages → review loop)
- Waiting times and resource requests
- Priority status (re-drawn at each stage)
- Clinical outcomes and flow counters
- RTT collection (cohort-filtered via `collect_stats`)


## 4.5 Workforce Resource and Resource Constraints

**Module:** `des/model/resources.py` — class `WorkforceHoursResource`

Custom  resource representing clinician capacity. Unlike standard SimPy resources, capacity is measured in **clinician hours** rather than server count.

| Feature | Behaviour |
|---------|----------|
| Capacity | Clinician **hours per weekday** (Mon–Fri); weekends = 0 |
| Queues | Priority deque served before standard |
| Scheduling | **Best-fit** — largest job fitting `hours_left` |
| Accounting | Released / used / unused hours |
| Validation | `final_validate()` — hour balance and queue conservation |


## 4.6 Autism Pathway System Class

**Module:** `des/model/system.py`

Responsible for:

- Weekday referral generation (exponential inter-arrivals)
- Patient routing and spawning
- One `WorkforceHoursResource` per clinical stage
- Daily queue-length snapshots to `Audit`
- System coordination via `run()` SimPy generator

### Simulation phases

```
Phase 1  WARM-UP     Days 0 → warmup_days           KPI cohort OFF
Phase 2  COLLECTION  Days warmup → + run_length      KPI cohort ON
Phase 3  DRAIN       After last referral until empty (max MAX_DRAIN_DAYS)
```


---

# 5. Classes and Functions of the Autism Pathway Model

| Component | Module | Role |
|-----------|--------|------|
| `Experiment` | `experiment.py` | Configuration, RNG, flow counters |
| `Patient` | `patient.py` | SimPy pathway process |
| `AutismPathwaySystem` | `system.py` | Arrivals, resources, coordination |
| `WorkforceHoursResource` | `resources.py` | Hour-based capacity engine |
| `Audit` | `audit.py` | RTT, wait, queue, utilisation KPIs |
| `single_run` / `multiple_runs` | `simulation.py` | Replication runners |
| Distributions | `distributions.py` | Exponential, Triangular, Bernoulli, Choice |
| Default parameters | `parameters.py` | Global scenario defaults |
| Provider scenario | `devon_parameters.py` | Devon (DAANA) published data |
| Verification | `verification.py` | Automated V&V test functions |
| calibration | — | *Planned (§7.4)* |
| State freeze | — | *Planned (§7.5)* |


## 6.1 Single Trace Run

Purpose: debugging, event verification, patient pathway tracing, resource behaviour inspection.

Set `parameters.TRACE = True` before calling `single_run()`. The `trace()` helper reads that flag at **call time** (so toggling it in the notebook works).

> Use a short `run_length` (e.g. 3 days) — trace output is verbose (one line per pathway event).


In [55]:
import des.model.parameters as params

params.TRACE = True
trace_exp = Experiment(auditor=Audit(), random_number_set=42)
trace_results = single_run(trace_exp, rep=0, warmup_days=0, run_length=3)
params.TRACE = False  # always turn off after tracing

print(
    f"Trace complete: {trace_results['ARRIVED_TOTAL']:.0f} cohort arrivals "
    f"in a 3-day collection window (see pathway messages above)"
)


[Time 0.221 | Monday] Patient 1 entered system. Referral submitted.
[Time 0.221 | Monday] Patient 1 Exit - Referral Rejected at Triage.
[Time 0.360 | Monday] Patient 2 entered system. Referral submitted.
[Time 0.360 | Monday] Patient 2 Exit - Referral Rejected at Triage.
[Time 0.499 | Monday] Patient 3 entered system. Referral submitted.
[Time 0.499 | Monday] Patient 3 queued for SCREENING (Queue pos: 0).
[Time 0.499 | Monday] >> Patient 3 officially ENTERED SCREENING stage.
[Time 0.527 | Monday] Patient 3 queued for PRE-ASSESSMENT (Queue pos: 0).
[Time 0.527 | Monday] >> Patient 3 officially ENTERED PRE-ASSESSMENT stage.
[Time 0.578 | Monday] Patient 3 queued for CORE ASSESSMENT (Queue pos: 0).
[Time 0.578 | Monday] >> Patient 3 officially ENTERED CORE ASSESSMENT stage.
[Time 0.740 | Monday] Patient 3 queued for POST-DIAG OTHER SUPPORT.
[Time 0.740 | Monday] >> Patient 3 officially ENTERED POST-DIAG OTHER SUPPORT stage.
[Time 0.809 | Monday] Patient 3 queued for FINAL CASE DISCHARGE R

## 6.2 Single Simulation Run

`single_run()` runs three phases in order:

```
Phase 1  WARM-UP     Days [0, warmup_days)                 collect_stats=False
Phase 2  COLLECTION  Days [warmup_days, warmup_days + run_length)   collect_stats=True  (KPI cohort)
Phase 3  DRAIN       After last referral until empty       (or max_drain_days reached)
```

This section covers **Part A — the drain period** first, then **Part B — warm-up**.

### Part A — With or without a drain period

**Why we need a drain:** referrals stop at the end of the collection window, but patients already in the pathway may still be waiting or in treatment. RTT and exit KPIs only include patients who **complete** during the run. If the simulation stops when collection ends (`max_drain_days=0`), slow patients remain in the system — `IN_SYSTEM_END > 0` and `COHORT_RTT_VALID=False`. Only fast-completing patients contribute to RTT, which **biases waits downward**.

**How it works:** after the collection window, `single_run` continues the SimPy clock in 30-day steps until every patient has exited **or** `max_drain_days` is reached (default `MAX_DRAIN_DAYS = 3650`). Check these flags:

| Flag | Meaning |
|------|---------|
| `COHORT_DRAIN_COMPLETE` | All cohort patients exited (`IN_SYSTEM_END == 0`) |
| `COHORT_RTT_VALID` | Safe to report diagnosis RTT for the cohort |
| `DRAIN_DAYS` | Simulation days spent in the drain phase |

```python
# Default — drain until empty (up to MAX_DRAIN_DAYS)
single_run(exp, warmup_days=0, run_length=1825)

# No drain — stops at collection end (RTT unreliable if patients remain)
single_run(exp, warmup_days=0, run_length=1825, max_drain_days=0)
```

> **When both columns look identical (`IN_SYSTEM_END=0`, `DRAIN_DAYS=0`):** capacity is high relative to demand — every cohort patient finishes **before** collection ends, so a drain phase is never needed. That is a special case. Part A below uses **stressed assessment capacity (10 h/day)** so `drain=0` leaves patients trapped and the difference is visible.


In [56]:
DEMO_RUN_LENGTH = 365 * 5  # 5-year collection window
DEFAULT_DRAIN = MAX_DRAIN_DAYS  # 3650 days — from parameters.py

# Stressed capacity for Part A only — ensures backlog outlasts the collection window.
# At ~18 h/day assessment, everyone may exit during collection and drain=0 looks fine.
DRAIN_DEMO_ASSESSMENT_H = 10.0


def _run(warmup_days=0, max_drain_days=DEFAULT_DRAIN, assessment_hours=None):
    """Helper: one replication; default drain unless overridden."""
    exp_kw = dict(
        auditor=Audit(),
        random_number_set=DEFAULT_RND_SET,
        use_fixed_seed=True,
    )
    if assessment_hours is not None:
        exp_kw["workforce_hours_assessment"] = assessment_hours
    return single_run(
        Experiment(**exp_kw),
        rep=0,
        warmup_days=warmup_days,
        run_length=DEMO_RUN_LENGTH,
        max_drain_days=max_drain_days,
    )


# --- Part A: drain = 0 vs default drain (stressed capacity, no warm-up) ---
no_drain = _run(warmup_days=0, max_drain_days=0, assessment_hours=DRAIN_DEMO_ASSESSMENT_H)
full_drain = _run(warmup_days=0, max_drain_days=DEFAULT_DRAIN, assessment_hours=DRAIN_DEMO_ASSESSMENT_H)

drain_keys = [
    "MAX_DRAIN_DAYS",
    "COLLECTION_END_DAYS",
    "SIM_END_DAYS",
    "DRAIN_DAYS",
    "IN_SYSTEM_END",
    "IN_SYSTEM_ALL",
    "COHORT_DRAIN_COMPLETE",
    "COHORT_RTT_VALID",
    "ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS",
    "KPI_SAMPLE_N_RTT_DIAGNOSIS",
    "FLOW_DIAGNOSIS_CONFIRMED",
    "FLOW_DIAGNOSIS_RATE_PCT",
]
drain_df = pd.DataFrame(
    {
        "drain=0 (stops at collection end)": {k: no_drain[k] for k in drain_keys},
        f"default drain (max {DEFAULT_DRAIN:.0f} d)": {k: full_drain[k] for k in drain_keys},
    }
)

print(
    f"Part A — drain = 0 vs default drain "
    f"(assessment={DRAIN_DEMO_ASSESSMENT_H} h/day, no warm-up, seed 42)"
)
display(drain_df.round(2))





Part A — drain = 0 vs default drain (assessment=10.0 h/day, no warm-up, seed 42)


,drain=0 (stops at collection end),default drain (max 3650 d)
MAX_DRAIN_DAYS,0,3650
COLLECTION_END_DAYS,1825.0,1825.0
SIM_END_DAYS,1825.0,2845.0
DRAIN_DAYS,0.0,1020.0
IN_SYSTEM_END,1433,0
IN_SYSTEM_ALL,1433,0
COHORT_DRAIN_COMPLETE,False,True
COHORT_RTT_VALID,False,True
ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS,280.3341,484.251116
KPI_SAMPLE_N_RTT_DIAGNOSIS,1944,2966


### Part B — With or without a warm-up period

**Why we need warm-up:** starting from an empty system (queues = 0) is unrealistic — real services carry backlog. Warm-up fills queues and stabilises utilisation **before** KPI collection begins.

**How it works:** set `warmup_days=0` for an empty start, or e.g. `warmup_days=365` for one year of pre-cohort referrals. Warm-up patients still **compete for capacity** with cohort patients but are **excluded from RTT KPIs** (`collect_stats=False`).

| Counter / KPI | No warm-up | With warm-up |
|---------------|------------|--------------|
| `ARRIVED_ALL` | = cohort only | warm-up **+** cohort referrals |
| `ARRIVED_TOTAL` | cohort only | cohort only |
| RTT / wait KPIs | cohort only | cohort only |
| `COHORT_DRAIN_COMPLETE` | should be `True` (with default drain) | should be `True` |

Collection window: `[warmup_days, warmup_days + run_length)`.

> Always use **default drain** when comparing warm-up scenarios — otherwise trapped patients invalidate RTT (see Part A).


In [57]:
WARMUP_COMPARE_DAYS = 365  # 1-year warm-up (default in parameters.py is 730)

# --- Part B: no warm-up vs warm-up (both with default drain) ---
baseline = _run(warmup_days=0)
with_warmup = _run(warmup_days=WARMUP_COMPARE_DAYS)

warmup_keys = [
    "WARMUP_DAYS",
    "ARRIVED_ALL",
    "ARRIVED_TOTAL",
    "FLOW_DIAGNOSIS_CONFIRMED",
    "FLOW_DIAGNOSIS_RATE_PCT",
    "ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS",
    "QUEUE_PEAK_ANY_STAGE",
    "IN_SYSTEM_END",
    "COHORT_DRAIN_COMPLETE",
    "COHORT_RTT_VALID",
    "DRAIN_DAYS",
]
warmup_df = pd.DataFrame(
    {
        "no warm-up": {k: baseline[k] for k in warmup_keys},
        f"{WARMUP_COMPARE_DAYS}d warm-up": {k: with_warmup[k] for k in warmup_keys},
    }
)
warmup_df["delta"] = warmup_df[f"{WARMUP_COMPARE_DAYS}d warm-up"] - warmup_df["no warm-up"]

print(f"Part B — no warm-up vs {WARMUP_COMPARE_DAYS}d warm-up (default drain, seed 42)")
display(warmup_df.round(2))




Part B — no warm-up vs 365d warm-up (default drain, seed 42)


,no warm-up,365d warm-up,delta
WARMUP_DAYS,0.0,365.0,365.0
ARRIVED_ALL,6609,7895,1286
ARRIVED_TOTAL,6609,6588,-21
FLOW_DIAGNOSIS_CONFIRMED,2966,2973,7
FLOW_DIAGNOSIS_RATE_PCT,44.878196,45.127505,0.249308
ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS,484.251116,682.593232,198.342115
QUEUE_PEAK_ANY_STAGE,1433.0,1714.0,281.0
IN_SYSTEM_END,0,0,0
COHORT_DRAIN_COMPLETE,True,True,0
COHORT_RTT_VALID,True,True,0


## 6.3 Multiple Replications (Fixed and Random Seeds)

`multiple_runs()` calls `single_run()` `n_reps` times (deep-copying the experiment each time) and returns one DataFrame row per replication.

**Fixed seed** (`use_fixed_seed=True`): each rep uses `base_seed + rep` — reproducible, low variance between reps (same stochastic stream offsets).

**Random seeds** (`use_fixed_seed=False`): independent seed per rep — captures run-to-run variability for confidence intervals.

Both examples below use `warmup_days=0`, default drain, and the same 5-year collection window as §6.2.


In [58]:
DEMO_RUN_LENGTH = 365 * 5  # 5-year collection window (same as §6.2)
N_REPS = 5

experiment = Experiment(
    auditor=Audit(),
    random_number_set=DEFAULT_RND_SET,
    use_fixed_seed=True,
)
results = multiple_runs(
    experiment,
    n_reps=N_REPS,
    warmup_days=0,
    run_length=DEMO_RUN_LENGTH,
    n_jobs=1,
)

summary_keys = [
    "ARRIVED_ALL",
    "ARRIVED_TOTAL",
    "FLOW_DIAGNOSIS_CONFIRMED",
    "FLOW_DIAGNOSIS_RATE_PCT",
    "ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS",
    "QUEUE_PEAK_ANY_STAGE",
    "OVERALL_SYSTEM_UTILISATION",
    "COHORT_DRAIN_COMPLETE",
    "COHORT_RTT_VALID",
]

print(f"Basic multiple_runs — {N_REPS} replications, fixed seed, warmup_days=0")
display(results[summary_keys].round(2))

pd.DataFrame(results[summary_keys].mean().round(2).rename("mean_across_reps"))


Basic multiple_runs — 5 replications, fixed seed, warmup_days=0


,ARRIVED_ALL,ARRIVED_TOTAL,FLOW_DIAGNOSIS_CONFIRMED,FLOW_DIAGNOSIS_RATE_PCT,ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS,QUEUE_PEAK_ANY_STAGE,OVERALL_SYSTEM_UTILISATION,COHORT_DRAIN_COMPLETE,COHORT_RTT_VALID
0,6609,6609,2966,44.88,484.25,1433.0,33.62,True,True
1,6509,6509,2961,45.49,474.57,1391.0,33.91,True,True
2,6531,6531,2941,45.03,457.98,1341.0,33.75,True,True
3,6690,6690,2975,44.47,496.51,1447.0,33.74,True,True
4,6711,6711,2987,44.51,516.76,1471.0,33.67,True,True


,mean_across_reps
ARRIVED_ALL,6610.00
ARRIVED_TOTAL,6610.00
FLOW_DIAGNOSIS_CONFIRMED,2966.00
FLOW_DIAGNOSIS_RATE_PCT,44.88
ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS,486.02
QUEUE_PEAK_ANY_STAGE,1416.60
OVERALL_SYSTEM_UTILISATION,33.74
COHORT_DRAIN_COMPLETE,1.00
COHORT_RTT_VALID,1.00


In [59]:
# Fixed vs random seed — compare mean and std across replications
exp_fixed = Experiment(auditor=Audit(), random_number_set=DEFAULT_RND_SET, use_fixed_seed=True)
df_fixed = multiple_runs(
    exp_fixed,
    n_reps=N_REPS,
    warmup_days=0,
    run_length=DEMO_RUN_LENGTH,
    n_jobs=1,
    use_fixed_seed=True,
)

exp_random = Experiment(auditor=Audit(), random_number_set=DEFAULT_RND_SET, use_fixed_seed=True)
df_random = multiple_runs(
    exp_random,
    n_reps=N_REPS,
    warmup_days=0,
    run_length=DEMO_RUN_LENGTH,
    n_jobs=1,
    use_fixed_seed=False,
)

compare_cols = [
    "ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS",
    "FLOW_DIAGNOSIS_RATE_PCT",
    "QUEUE_PEAK_ANY_STAGE",
    "ARRIVED_TOTAL",
    "FLOW_DIAGNOSIS_CONFIRMED",
    "OVERALL_SYSTEM_UTILISATION",
]

print("Fixed vs random seed — mean and std across replications")
pd.DataFrame(
    {
        "fixed_seed_mean": df_fixed[compare_cols].mean(),
        "fixed_seed_std": df_fixed[compare_cols].std(),
        "random_seed_mean": df_random[compare_cols].mean(),
        "random_seed_std": df_random[compare_cols].std(),
    }
).round(2)


Fixed vs random seed — mean and std across replications


,fixed_seed_mean,fixed_seed_std,random_seed_mean,random_seed_std
ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS,486.02,22.22,475.87,21.94
FLOW_DIAGNOSIS_RATE_PCT,44.88,0.42,44.87,0.81
QUEUE_PEAK_ANY_STAGE,1416.60,51.29,1371.80,50.68
ARRIVED_TOTAL,6610.00,90.89,6513.20,83.51
FLOW_DIAGNOSIS_CONFIRMED,2966.00,17.12,2923.00,76.58
OVERALL_SYSTEM_UTILISATION,33.74,0.11,33.72,0.14


## 7.1 Parameter Experiments

Investigate referral demand, capacity, and branching via `Experiment(**kwargs)`.


In [60]:
low = single_run(
    Experiment(auditor=Audit(), random_number_set=42, iat=1 / REFERRALS_PER_DAY),
    rep=0, warmup_days=0, run_length=DEMO_RUN_LENGTH,
)
high = single_run(
    Experiment(auditor=Audit(), random_number_set=42, iat=1 / (REFERRALS_PER_DAY * 2)),
    rep=0, warmup_days=0, run_length=DEMO_RUN_LENGTH,
)
extra_cap = single_run(
    Experiment(auditor=Audit(), random_number_set=42, workforce_hours_assessment=36.0),
    rep=0, warmup_days=0, run_length=DEMO_RUN_LENGTH,
)

# Keys must match Audit.summarize() output (CAPACITY_UTILISATION_* not *_UTILISATION)
compare_keys = [
    "ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS",
    "QUEUE_PEAK_ANY_STAGE",
    "OVERALL_SYSTEM_UTILISATION",
    "CAPACITY_UTILISATION_SCREENING",
    "CAPACITY_UTILISATION_ASSESSMENT",
]

def _compare_row(results):
    row = {}
    for k in compare_keys:
        if k not in results:
            raise KeyError(f"{k!r} not in single_run results — check Audit.summarize() key names")
        v = results[k]
        # Stage utilisation is 0–1; overall is already percent
        if k.startswith("CAPACITY_UTILISATION_"):
            v = v * 100
        row[k] = v
    return row

pd.DataFrame(
    {
        f"{REFERRALS_PER_DAY:.0f} rpd": _compare_row(low),
        f"{REFERRALS_PER_DAY*2:.0f} rpd": _compare_row(high),
        "+assessment hours": _compare_row(extra_cap),
    }
).round(2)


,5 rpd,10 rpd,+assessment hours
ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS,484.25,1831.81,0.40
QUEUE_PEAK_ANY_STAGE,1433.00,4779.00,7.00
OVERALL_SYSTEM_UTILISATION,33.62,34.22,35.14
CAPACITY_UTILISATION_SCREENING,38.75,40.47,60.36
CAPACITY_UTILISATION_ASSESSMENT,86.39,87.21,37.39


## 7.3 Sceranio analysis using multiple run 

Compare sceranio like deman increase, capactity increse.

In [61]:
# Multiple-run scenario comparison — mean KPIs across replications

DEMO_RUN_LENGTH = 365 * 5
N_REPS = 5

baseline = multiple_runs(
    Experiment(auditor=Audit(), random_number_set=42),
    n_reps=N_REPS,
    warmup_days=0,
    run_length=DEMO_RUN_LENGTH,
    n_jobs=1,
)

high_demand = multiple_runs(
    Experiment(auditor=Audit(), random_number_set=42, iat=1 / (REFERRALS_PER_DAY * 2)),
    n_reps=N_REPS,
    warmup_days=0,
    run_length=DEMO_RUN_LENGTH,
    n_jobs=1,
)

high_capacity = multiple_runs(
    Experiment(auditor=Audit(), random_number_set=42, workforce_hours_assessment=36.0),
    n_reps=N_REPS,
    warmup_days=0,
    run_length=DEMO_RUN_LENGTH,
    n_jobs=1,
)

compare_keys = [
    "ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS",
    "QUEUE_PEAK_ANY_STAGE",
    "CAPACITY_UTILISATION_ASSESSMENT",
    "FLOW_DIAGNOSIS_RATE_PCT",
    "FLOW_DIAGNOSIS_CONFIRMED",
    "OVERALL_SYSTEM_UTILISATION",
]


def _compare_row(df):
    """Mean KPIs across replication rows returned by multiple_runs()."""
    row = {}
    for k in compare_keys:
        if k not in df.columns:
            raise KeyError(f"{k!r} not in multiple_runs results — check Audit.summarize() key names")
        v = df[k].mean()
        # Stage utilisation is 0–1; overall is already percent
        if k.startswith("CAPACITY_UTILISATION_"):
            v = v * 100
        row[k] = v
    return row

results_df = pd.DataFrame(
    {
        "baseline": _compare_row(baseline),
        "high_demand": _compare_row(high_demand),
        "high_capacity": _compare_row(high_capacity),
    }
)

results_df.round(2)


,baseline,high_demand,high_capacity
ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS,486.02,1836.37,0.40
QUEUE_PEAK_ANY_STAGE,1416.60,4740.60,8.20
CAPACITY_UTILISATION_ASSESSMENT,86.48,87.18,36.93
FLOW_DIAGNOSIS_RATE_PCT,44.88,43.79,44.88
FLOW_DIAGNOSIS_CONFIRMED,2966.00,5758.60,2966.00
OVERALL_SYSTEM_UTILISATION,33.74,34.14,34.79


## 7.3 Warm-up and Steady-State Analysis

Compare empty system, warm-up, and operational steady state.


In [62]:
warmup_rows = []
for label, warmup in [("empty start", 0), ("1-year warm-up", 365), ("2-year warm-up", 730)]:
    res = single_run(
        Experiment(auditor=Audit(), random_number_set=42),
        rep=0, warmup_days=warmup, run_length=DEMO_RUN_LENGTH,
    )
    warmup_rows.append(
        {
            "scenario": label,
            "warmup_days": warmup,
            "diagnosis_rtt_days": res["ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS"],
            "peak_queue": res["QUEUE_PEAK_ANY_STAGE"],
            "utilisation_pct": res["OVERALL_SYSTEM_UTILISATION"],
        }
    )
pd.DataFrame(warmup_rows).round(2)


,scenario,warmup_days,diagnosis_rtt_days,peak_queue,utilisation_pct
0,empty start,0,484.25,1433.0,33.62
1,1-year warm-up,365,682.59,1714.0,33.39
2,2-year warm-up,730,873.96,2014.0,32.98


## 7.4 Dynamic Calibration

**Status: planned.** Will automatically calibrate provider models by matching operational targets (RTT, queue size, workforce utilisation).

**Current approach:** manual calibration via `devon_parameters.py` — scale workforce hours until simulated RTT matches published ICB statistics (see §7.6).


## 7.5 Freeze-State Initialisation

**Status: planned.** Will capture a simulation snapshot at operational steady state and restore it for scenario analysis.

**Current workaround:** use `warmup_days=730` (or longer) to build a realistic backlog before the collection window.


---

# 8. Model Verification and Validation

## 8.1 Internal Verification



In [63]:

from des.model.verification import (
    run_demand_stress_verification,
    run_flow_conservation_verification,
    run_math_convergence_verification,
    run_rtt_cohort_verification,
    run_seed_verification,
)

run_seed_verification()
run_flow_conservation_verification()
run_rtt_cohort_verification()
run_demand_stress_verification()
run_math_convergence_verification()


 SUITE: 1. SEED CONTROL & REPRODUCIBILITY VERIFICATION
 -> SUCCESS: Fixed seeds are deterministic and reproducible.

 SUITE: 2. PATIENT FLOW MASS-CONSERVATION VERIFICATION
  Arrived: 6509 | Exited: 6509 | In system: 0
 -> SUCCESS: Mass balance verified (arrivals = exits + in system).

 SUITE: 3. RTT COHORT COMPLETENESS VERIFICATION
  Diagnosis RTT samples: 2961 | Diagnoses confirmed: 2961 | In system: 0
 -> SUCCESS: Cohort RTT computed after full drain (zero backlog).

 SUITE: 5. DEMAND STRESS TEST
   5 rpd: peak queue=283 | diagnosis RTT=92.6 d | utilisation=32.6%
  10 rpd: peak queue=944 | diagnosis RTT=375.5 d | utilisation=33.8%
  20 rpd: peak queue=2,518 | diagnosis RTT=933.3 d | utilisation=33.9%
 -> SUCCESS: Queue and RTT increase with demand; utilisation shows expected stress response.

 SUITE: 4. WORKFORCE-HOURS MATHEMATICAL CONVERGENCE
  Assessment RTT: 0.291 d | Diagnosis RTT: 0.371 d | Delta: 0.055
 -> SUCCESS: Infinite-capacity scheduler validation passed.




#
## 8.2 Dynamic Calibration Verification

**Status: planned** — will verify convergence, target matching, and snapshot consistency.

## 8.3 External Validation (Future)

Compare model outputs against NHS provider operational data (RTT, queues, utilisation, throughput).
Devon scenario (§7.6) is a first step toward external validation using published statistics.


---

# 9. Future Work

## 9.1 Provider Calibration Using NHS Operational Data

Use operational NHS provider data to calibrate arrival rates, capacity, branching probabilities, and service durations. Extend `devon_parameters.py` pattern to other ICBs.

## 9.2 Intervention Analysis

Evaluate operational improvements before implementation: additional clinicians, increased clinics, demand management, service redesign.

## 9.3 Model Optimisation

Potential future work: Simulation-Based Optimisation (SBO), multi-parameter calibration, Bayesian optimisation, parallel calibration.

## 9.4 Decision Support System

Potential deployment as an interactive dashboard, provider planning tool, or NHS service planning platform.

